## kiểm tra tổng số điều, khoản, điểm


In [ ]:
import json

with open("/content/drive/MyDrive/IE103/Project/final_chunked.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("Total nodes:", len(data))

from collections import Counter

print(Counter(x["level"] for x in data))

Total nodes: 55637
Counter({'clause': 26529, 'point': 21019, 'article': 8089})


# kiểm tra tổng số đoạn luật sẽ được indexing để đưa vào vector database


In [ ]:
import json

with open("/content/drive/MyDrive/IE103/Project/final_chunked.json", "r", encoding="utf-8") as f:
    all_nodes = json.load(f)

clause_ids_with_point = set(
    node["parent_id"]
    for node in all_nodes
    if node["level"] == "point" and node["parent_id"]
)

def should_index(node):
    level = node["level"]

    if level == "article":
        return False

    if level == "point":
        return True

    if level == "clause":
        return node["id"] not in clause_ids_with_point

    return False

index_nodes = [n for n in all_nodes if should_index(n)]

print(len(index_nodes))

41953


In [ ]:
import json

with open("/content/drive/MyDrive/IE103/Project/final_chunked.json", "r", encoding="utf-8") as f:
    all_nodes = json.load(f)

clause_ids_with_point = set(
    node["parent_id"]
    for node in all_nodes
    if node["level"] == "point" and node["parent_id"]
)

def should_index(node):
    level = node["level"]

    if level == "article":
        return False

    if level == "point":
        return True

    if level == "clause":
        return node["id"] not in clause_ids_with_point

    return False

index_nodes = [n for n in all_nodes if should_index(n)]

rows = []

for node in index_nodes:

    article_title = node["metadata"].get(
        "article_title", ""
    )

    context = node.get(
        "context", ""
    )

    content = node.get(
        "content", ""
    )

    embed_text = " ".join(
        x.strip()
        for x in [article_title, context, content]
        if x and x.strip()
    )

    rows.append({
        "node_id": node["id"],
        "level": node["level"],
        "article_id": node["metadata"].get("article_id"),
        "article_title": article_title,
        "article_index": node["metadata"].get("article_index"),
        "parent_id": node.get("parent_id"),
        "content": content,
        "embed_text": embed_text
    })

with open(
    "pgvector_dataset.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        rows,
        f,
        ensure_ascii=False,
        indent=2
    )

print(len(rows))

41953


In [ ]:
print(rows[0])

{'node_id': '135_VBHN-VPQH_D2_K1', 'level': 'clause', 'article_id': '135_VBHN-VPQH_D2', 'article_title': 'Cơ sở của trách nhiệm hình sự', 'article_index': 2, 'parent_id': '135_VBHN-VPQH_D2', 'content': 'Chỉ người nào phạm một tội đã được Bộ luật Hình sự quy định mới phải chịu trách nhiệm hình sự.', 'embed_text': 'Cơ sở của trách nhiệm hình sự Chỉ người nào phạm một tội đã được Bộ luật Hình sự quy định mới phải chịu trách nhiệm hình sự.'}


In [ ]:
import json

with open("pgvector_dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

article_ids = set()

for row in data:
    article_ids.add(row["article_id"])

print("Số article_id duy nhất:", len(article_ids))
print("Số chunk:", len(data))

Số article_id duy nhất: 7003
Số chunk: 41953


# Xuất file csv cho 2 bảng để lưu vào postgreSQL


*   legal_articles.csv
*   legal_chunks.csv



In [ ]:
import json
import pandas as pd

with open("pgvector_dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# ==========================
# legal_articles
# ==========================
articles = {}

for row in data:
    aid = row["article_id"]

    if aid not in articles:
        articles[aid] = {
            "article_id": aid,
            "article_title": row["article_title"],
            "article_index": row["article_index"]
        }

articles_df = pd.DataFrame(
    articles.values()
)

articles_df.to_csv(
    "legal_articles.csv",
    index=False,
    encoding="utf-8-sig"
)

# ==========================
# legal_chunks
# ==========================
chunks_df = pd.DataFrame([
    {
        "node_id": row["node_id"],
        "article_id": row["article_id"],
        "level": row["level"],
        "parent_id": row["parent_id"],
        "content": row["content"],
        "embed_text": row["embed_text"]
    }
    for row in data
])

chunks_df.to_csv(
    "legal_chunks.csv",
    index=False,
    encoding="utf-8-sig"
)

print("articles:", len(articles_df))
print("chunks:", len(chunks_df))

articles: 7003
chunks: 41953
